In [0]:
%run ../functions/functions

In [0]:
# Nome do banco de dados onde a tabela será salva
database_name = "dimensao"

# Nome da tabela de destino
table_name = "dm_ncm"

# Caminho alvo no formato database.tabela
target_path = f"{database_name}.{table_name}"

# Nome da chave primária da tabela
pk = "SK_NCM"

In [0]:
# Caminho para os dados silver da tabela NCM
silver_path_s = f"abfss://silver@stgbbb.dfs.core.windows.net/balancacomercial/NCM/"

# Caminho para os dados silver da tabela NCM_SH_CONSOLIDADA
silver_path_sh = f"abfss://silver@stgbbb.dfs.core.windows.net/balancacomercial/NCM_SH_CONSOLIDADA/"

In [0]:
# Lê os dados do caminho Delta referente à tabela NCM e armazena em um DataFrame
df_s = spark.read.format("delta").load(silver_path_s)

# Lê os dados do caminho Delta referente à tabela NCM_SH_CONSOLIDADA e armazena em um DataFrame
df_sh = spark.read.format("delta").load(silver_path_sh)

In [0]:
# Cria uma view temporária chamada "df_ncm" a partir do DataFrame df_s
df_s.createOrReplaceTempView("df_ncm")
# Cria uma view temporária chamada "df_ncm_sh" a partir do DataFrame df_sh
df_sh.createOrReplaceTempView("df_ncm_sh")

In [0]:
query = """
-- Seleciona colunas principais da tabela NCM e faz junção com tabela SH consolidada
select 
  c.SK_NCM,         -- Chave substituta da NCM
  c.CO_NCM,         -- Código NCM
  c.NO_NCM_POR,     -- Nome NCM em português
  s.CO_SH4,         -- Código SH4 da tabela consolidada
  c.CO_SH6          -- Código SH6 da tabela NCM
from df_ncm as c
left join df_ncm_sh as s 
  on try_cast(c.CO_SH6 as BIGINT) = try_cast(s.CO_SH6 as BIGINT) -- Junção pelo código SH6 convertido para BIGINT
"""

In [0]:
# Executa a query SQL definida anteriormente e armazena o resultado no DataFrame df_final
df_final = spark.sql(query)

In [0]:
# Salva o DataFrame df_final como uma tabela Hive no caminho especificado, utilizando a chave primária pk
save_hive_table(df_final, target_path, pk)